# 03. Bóc Tách Trích Dẫn BibTeX, Hợp Nhất Đa Phiên Bản & Đánh Giá Chất Lượng (BibTeX Extraction & Unified Evaluation)

> **Môn học:** Nhập môn Khoa học Dữ liệu (Introduction to Data Science) — HCMUS  
> **Milestone 2:** Bóc tách Cấu trúc Tài liệu & Chuẩn Hóa Phân Cấp  
> **Các thành viên phụ trách:** Cáp Hửu Duy (BibTeX & Dedup Trích dẫn), Nguyễn Lê Anh Tuấn (Tổng hợp Đánh giá & Báo cáo)  
> **Mục tiêu:** Bóc tách tệp BibTeX độc lập, chuyển đổi `\bibitem` inline thành bản ghi chuẩn, hợp nhất trích dẫn đa phiên bản bằng thuật toán gom cụm bắc cầu, và đo lường định lượng trên Bộ chuẩn vàng tự nhiên.

---

### Thách Thức Kỹ Thuật Trong Xử Lý Trích Dẫn Học Thuật

Trong mã nguồn LaTeX của arXiv, trích dẫn học thuật tồn tại ở hai trường phái:
1. **Tệp `.bib` có cấu trúc:** Chiếm ~59.5% số version, có thể bóc tách dễ dàng qua cú pháp chuẩn.
2. **Khai báo `\bibitem` tự do (Free-text):** Chiếm ~17.2% số version (~7.000 version toàn nhóm), tác giả tự gõ tay không theo chuẩn ngữ pháp cố định.

Đề bài yêu cầu: *"For citations defined using `\bibitem` within the `.tex` files, students must convert them into standard BibTeX entries using programmatic tools such as regular expressions."*

In [1]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data_dir = Path("../data") if Path("../data").exists() else Path("data")

def load_or_placeholder(filename: str, desc: str):
    filepath = data_dir / filename
    if not filepath.exists():
        print(f"⏳ [Đang xử lý] Chưa có tệp {filename} — {desc}.")
        return None
    df = pd.read_csv(filepath)
    print(f"✅ Đã nạp thành công {filename}: {len(df):,} bản ghi ({desc}).")
    return df

print("______________________________________________________________________________")
print("KHỞI TẠO DỮ LIỆU ĐÁNH GIÁ TRÍCH DẪN & HỢP NHẤT TOÀN DIỆN:")
df_bib = load_or_placeholder("bib.csv", "Nhật ký bóc tách trích dẫn từng version (Hửu Duy)")
df_bib_dedup = load_or_placeholder("bib_dedup.csv", "Nhật ký hợp nhất trích dẫn theo publication (Hửu Duy)")
print("______________________________________________________________________________")

______________________________________________________________________________
KHỞI TẠO DỮ LIỆU ĐÁNH GIÁ TRÍCH DẪN & HỢP NHẤT TOÀN DIỆN:
⏳ [Đang xử lý] Chưa có tệp bib.csv — Nhật ký bóc tách trích dẫn từng version (Hửu Duy).
⏳ [Đang xử lý] Chưa có tệp bib_dedup.csv — Nhật ký hợp nhất trích dẫn theo publication (Hửu Duy).
______________________________________________________________________________


### 1. Kỹ Thuật Bóc Tách `\bibitem` Viết Tay Bằng Biểu Thức Chính Quy (Regex Parser)

Module `src/parser/bib.py` do **Cáp Hửu Duy** phụ trách triển khai:
- `split_bibitems(tex: str)`: Tách các khối trích dẫn dựa trên mẫu regex `\\bibitem\s*(?:\[[^\]]*\])?\s*\{([^}]+)\}`.
- `parse_bibitem(key: str, raw: str)`: Bóc tách các trường thông tin từ văn bản thô:
  - `year`: Tìm kiếm số có 4 chữ số trong khoảng 19xx–20xx.
  - `doi`: Regex nhận diện chuỗi định danh số `10.\d{4,9}/...`.
  - `arxiv_id`: Bắt cả hai chuẩn mã arXiv: chuẩn mới (`2310.12345`) và chuẩn cũ trước năm 2007 (`hep-th/9901001`).
  - `title`: Khai thác các mẫu nhấn mạnh `\emph{...}`, dấu ngoặc kép `''...''` hoặc `"..."`.
  - **Bảo tồn cột `raw`:** Giữ nguyên chuỗi gốc để hỗ trợ kiểm tra và đối chiếu.

In [2]:
# Mô phỏng bóc tách trích dẫn \bibitem thực tế
import re

sample_bibitem = r"""
\bibitem{vaswani2017attention}
A. Vaswani, N. Shazeer, N. Parmar, J. Uszkoreit, L. Jones, A. N. Gomez, L. Kaiser, and I. Polosukhin,
``Attention is all you need,'' in \emph{Advances in Neural Information Processing Systems}, 2017, arXiv:1706.03762.
"""

def demo_parse_bibitem(raw_text):
    key_match = re.search(r'\\bibitem\s*(?:\[[^\]]*\])?\s*\{([^}]+)\}', raw_text)
    citation_key = key_match.group(1) if key_match else "unknown"
    year_match = re.search(r'\b(19|20)\d{2}\b', raw_text)
    year = int(year_match.group(0)) if year_match else None
    arxiv_match = re.search(r'arXiv[:\s]*(\d{4}\.\d{4,5}|[a-z-]+/\d{7})', raw_text, re.I)
    arxiv_id = arxiv_match.group(1) if arxiv_match else None
    title_match = re.search(r"``(.+?)''", raw_text)
    title = title_match.group(1).strip() if title_match else None
    
    return {
        "citation_key": citation_key,
        "title": title,
        "year": year,
        "arxiv_id": arxiv_id
    }

parsed_res = demo_parse_bibitem(sample_bibitem)
print("KẾT QUẢ BÓC TÁCH MỤC TRÍCH DẪN \\bibitem:")
print("______________________________________________________________________________")
for k, v in parsed_res.items():
    print(f"- {k:<15}: {v}")

KẾT QUẢ BÓC TÁCH MỤC TRÍCH DẪN \bibitem:
______________________________________________________________________________
- citation_key   : vaswani2017attention
- title          : Attention is all you need,
- year           : 2017
- arxiv_id       : 1706.03762


### 2. Thuật Toán Hợp Nhất Trích Dẫn Liên Phiên Bản & Đổi Tên `\cite`

Đề bài yêu cầu: *"unionize the fields rather than selecting one entry while discarding all other information"*.

Khi một tài liệu xuất hiện ở nhiều version với citation key khác nhau (ví dụ: v1 dùng `vaswani2017`, v2 đổi thành `vit_transformer`):
1. **Chữ ký định danh (`signature`):** Nhận diện cùng một tài liệu thông qua `doi` hoặc chuỗi chuẩn hóa `(title_norm, authors_norm, year)`.
2. **Gom cụm bắc cầu (Transitive Clustering):** Nối nhóm tài liệu theo quan hệ bắc cầu để không bỏ sót các trường hợp thiếu năm hoặc khuyết DOI giữa các version.
3. **Quy tắc chọn key chuẩn:** Ưu tiên version nhỏ nhất $\rightarrow$ key ngắn nhất $\rightarrow$ thứ tự bảng chữ cái (đảm bảo tính tất định).
4. **Hợp nhất trường dữ liệu:** Lấy hợp (union) toàn bộ các trường: v1 có `year` + v2 có `doi` $\rightarrow$ bản ghi cuối cùng có đầy đủ cả `year` lẫn `doi`!
5. **Đổi tên `\cite` trong văn bản:** Chạy trước bước băm `element_id` để đảm bảo hai câu giống nhau ở v1 và v2 không bị lệch ID do khác tên citation key.

In [3]:
# Mô phỏng thuật toán hợp nhất thông tin hai bản ghi cùng tài liệu
entry_v1 = {"citation_key": "vaswani2017", "title": "Attention Is All You Need", "year": 2017, "doi": None}
entry_v2 = {"citation_key": "vit_transformer", "title": "Attention is all you need", "year": None, "doi": "10.5555/3295222"}

def merge_two_entries(e1, e2):
    return {
        "citation_key": min(e1["citation_key"], e2["citation_key"], key=len),
        "title": e1["title"],
        "year": e1["year"] or e2["year"],
        "doi": e1["doi"] or e2["doi"]
    }

merged = merge_two_entries(entry_v1, entry_v2)
print("KẾT QUẢ HỢP NHẤT TRÍCH DẪN (UNIONIZE FIELDS):")
print("______________________________________________________________________________")
print("Bản ghi v1:", entry_v1)
print("Bản ghi v2:", entry_v2)
print("Sau gộp   :", merged)
print("--> Giữ được cả Year từ v1 VÀ DOI từ v2!")

KẾT QUẢ HỢP NHẤT TRÍCH DẪN (UNIONIZE FIELDS):
______________________________________________________________________________
Bản ghi v1: {'citation_key': 'vaswani2017', 'title': 'Attention Is All You Need', 'year': 2017, 'doi': None}
Bản ghi v2: {'citation_key': 'vit_transformer', 'title': 'Attention is all you need', 'year': None, 'doi': '10.5555/3295222'}
Sau gộp   : {'citation_key': 'vaswani2017', 'title': 'Attention Is All You Need', 'year': 2017, 'doi': '10.5555/3295222'}
--> Giữ được cả Year từ v1 VÀ DOI từ v2!


### 3. Đánh Giá Khách Quan Trên Bộ Chuẩn Vàng Tự Nhiên (124 Version "Both")

Để kiểm định độ chính xác của bộ regex bóc tách mà không thiên vị:
- Nhóm tận dụng **124 version tồn tại song song cả file `.bib` lẫn `\bibitem`**.
- File `.bib` do tác giả cung cấp đóng vai trò là **Ground Truth**.
- Kết quả trích xuất từ regex được so khớp trực tiếp với Ground Truth trên 4 trường then chốt: `title`, `authors`, `year`, `doi`.
- Kết quả cho phép báo cáo trung thực và minh bạch tỷ lệ bóc tách thành công của module trước hội đồng chấm điểm.